####Paso 1 - Leer el archivo JSON usando "DataFrameReader de Spark"

In [0]:
%run "../includes/configurations"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movies_genre_schema = StructType(fields=[
  StructField("movieId", IntegerType(), True),
  StructField("genreId", IntegerType(), True)
])

In [0]:
movies_genre_df = spark.read \
  .schema(movies_genre_schema) \
  .json(f"{bronze_folder_path}/{v_file_date}/movie_genre.json")

In [0]:
movies_genre_df.printSchema()

In [0]:
display(movies_genre_df)

####Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_genre_final_df = movies_genre_df \
    .withColumnRenamed("movieId", "movie_id") \
    .withColumnRenamed("genreId", "genre_id") \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("enviroment", lit("Produccion")) \
    .withColumn("file_date", lit(v_file_date))

####Paso 3 - Escribir la salida en un formato "Parquet"

In [0]:
#overwrite_partition("movie_silver", "movies_genre", "file_date", v_file_date)

In [0]:
#movies_genre_final_df.write.mode("overwrite").partitionBy("movie_id").parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/movies_genre")
#movies_genre_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_genre")

merge_condition = 'tgt.movie_id = src.movie_id AND tgt.genre_id = src.genre_id AND tgt.file_date = src.file_date'
merge_delta_lake (movies_genre_final_df, "movie_silver", "movies_genre", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movies_genre
GROUP BY file_date;

In [0]:
%sql
SELECT *
FROM movie_silver.movies_genre;

In [0]:
#%fs
#ls abfss://silver@moviehistorybymg.dfs.core.windows.net/movies_genre

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/movies_genre"))